In [ ]:
import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
from matplotlib import pylab
import os
import sys
anndata2ri.activate()
import yaml
%reload_ext rpy2.ipython

from scipy.sparse import csr_matrix, isspmatrix


In [ ]:
pylab.rcParams['figure.figsize'] = (9, 9)
homeDir = os.getenv("HOME")
sys.path.insert(1, homeDir+"/utils/")
from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
from spatialUtils import *
from _DEAplots import *
from _Aggregation import *
from _plotting import *

In [ ]:
with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)

# Load data

In [ ]:
adata = sc.read_h5ad("./MelanomaCoCulture_processed.h5ad")
adata.layers["logCPU"] = adata.X.copy()

# Score signatures Bulk Derived

In [ ]:
from transferUtils import *

export_anndata_minimal(
    adata=adata,
    out_dir=f"./AUCready",
    base=f"VerrilloSC",
    layer="counts",                 # ignored for data, used only for shape checking
    obsm_key="X_pca",
    replace_counts_with_zeros=False, # <- zeros
)

# Load In House signature

In [ ]:
import os
import glob
import pandas as pd
import rapids_singlecell as rsc

topN = 300

BaseDir = "/data/projects/spatialTX/0_SignaturePrep/Contrasts_RAW_signatures_MB_MELA_NCRE_Neurons/"
signature_list = []

for filepath in glob.glob(os.path.join(BaseDir, "*.xlsx")):
    sig = pd.read_excel(filepath)

    # safer: split only the filename, not the full path
    fname = os.path.basename(filepath)
    sig_name = fname.split("_")[1]   # adjust index if needed after checking filenames

    # keep top N rows
    sig = sig[(sig["FDR"] < 0.01) & (sig["Celltypes_upregulated"] == sig_name) & (sig["genes"].isin(adata.var_names))].sort_values("logFC", ascending=False).head(topN)
    print(f"{len(sig)} Genes met requirements for {sig_name}")
    signature_list.append(sig)

signatureDF = pd.concat(signature_list, ignore_index=True)

In [ ]:
sc.set_figure_params(dpi_save=300, frameon=True, vector_friendly=True, fontsize=14, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='retina')
pylab.rcParams['figure.figsize'] = (9, 9)

In [ ]:
import numpy as np

# remove old score columns
old_cols = [col for col in adata.obs.columns if "scoregenes_" in col]
adata.obs.drop(columns=old_cols, inplace=True, errors="ignore")

# restore expression matrix
adata.X = adata.layers["logCPU"].copy()

# compute scores
rsc.get.anndata_to_GPU(adata)
rsc.pp.scale(adata, zero_center=False)

for celltype in signatureDF["celltype"].dropna().unique():
    genes = signatureDF.loc[signatureDF["celltype"] == celltype, "genes"].tolist()
    genes = [g for g in genes if g in adata.var_names]  # recommended
    
    if len(genes) == 0:
        print(f"Skipping {celltype}: no genes found in adata.var_names")
        continue

    sigName = f"scoregenes_{celltype}"
    rsc.tl.score_genes(adata, gene_list=genes, score_name=sigName)

rsc.get.anndata_to_CPU(adata)
adata.X = adata.layers["logCPU"].copy()

# collect NEW score columns after scoring
cols = [col for col in adata.obs.columns if "scoregenes_" in col]

if len(cols) == 0:
    raise ValueError("No scoregenes_ columns were created.")

q99List = [adata.obs[col].quantile(0.99) for col in cols]
q1list  = [adata.obs[col].quantile(0.01) for col in cols]

vmax = np.max(q99List)
vmin = np.min(q1list)

# sc.pl.umap(
#     adata,
#     color=cols,
#     size=50,
#     cmap="Greens",
#     vmax=vmax,
#     vmin=vmin
# )


sc.pl.umap(adata, color=cols,     vmax=vmax,
    vmin=vmin, cmap="Greens", size=50, add_outline=True, outline_width=(0.1, 0.00), save=f"SC_LongitudinalOrganoids_VerrilloSignature_scoregenes.png")

In [ ]:
%%R  -i homeDir 


library(igraph)
library(ggraph)

library(ggplot2)
library(dplyr)

source(paste0(homeDir,"/utils/transferUtils.R"))

sce <- load_export_as_sce(
  out_dir = "./AUCready",
  base = "VerrilloSC",
  reduced_name = "X_pca",
  add_coords_to_reduced = TRUE,
  include_coords_in_coldata = TRUE,
  counts_transpose = TRUE,
  sep = "\t"
)


exprMatrix  <- assay(sce, "counts")
dim(exprMatrix )

## AUCELL

In [ ]:
%%R  -i homeDir -i signatureDF -w 1500 -h 1500 -o df_auc0

library(GSEABase)
library(AUCell)
as_gene_vector <- function(x) {
  if (is.null(x)) return(character(0))
  x <- as.character(x)
  x <- trimws(x)
  x <- x[!is.na(x) & nzchar(x)]
  unique(x)
}

# split long table -> list of character vectors
Signatures <- split(signatureDF$gene, signatureDF$celltype)
Signatures <- lapply(Signatures, as_gene_vector)

# drop empty sets
Signatures <- Signatures[lengths(Signatures) > 0]

# build GeneSetCollection
geneSets <- GeneSetCollection(
  mapply(
    function(ids, nm) GeneSet(geneIds = ids, setName = nm),
    ids = Signatures,
    nm  = names(Signatures),
    SIMPLIFY = FALSE
  )
)

# quick check
length(geneSets)
names(geneSets)[1:min(10, length(geneSets))]


setClassUnion("ExpData", c("matrix", "SummarizedExperiment"))

# Run aucell
cells_rankings <- AUCell_buildRankings(exprMatrix, plotStats=TRUE)
cells_AUC <- AUCell_calcAUC(geneSets, cells_rankings)


set.seed(333)
par(mfrow=c(5,5)) 
cells_assignment <- AUCell_exploreThresholds(cells_AUC, plotHist=TRUE, assign=TRUE)

selectedThresholds <- getThresholdSelected(cells_assignment)
auc_mat <- getAUC(cells_AUC)   # rows = signatures, cols = barcodes

# keep only signatures present in auc_mat
selectedThresholds <- selectedThresholds[intersect(names(selectedThresholds), rownames(auc_mat))]

scorelist_list <- list()

for (geneSetName in names(selectedThresholds)) {
  auc_vec <- auc_mat[geneSetName, ]
  thr <- selectedThresholds[[geneSetName]]

  #auc_vec[auc_vec <= thr] <- 0  # Option B: AUC if pass else 0
  scorelist_list[[paste0("AUC_score_", geneSetName)]] <- auc_vec
}

# barcode as rownames (index), not a column
df_auc0 <- as.data.frame(scorelist_list, check.names = FALSE)

In [ ]:
for i in df_auc0.columns:
    if i in adata.obs.columns:
        del adata.obs[i]

adata.obs = pd.concat([adata.obs, df_auc0], axis = 1)
cols = [c for c in adata.obs.columns if c.startswith("AUC_score")]


q99List = []
q1list = []
for scorecol in cols:
    print(scorecol)
    q99List.append(adata.obs[scorecol].quantile(0.99))
    q1list.append(adata.obs[scorecol].quantile(0.01))
vmax = np.max(q99List)  
vmin = np.max(q1list)  



sc.pl.umap(adata, color=cols,     vmax=vmax,
    vmin=vmin, cmap="Greens", size=50, add_outline=True, outline_width=(0.1, 0.00), , save=f"SC_LongitudinalOrganoids_VerrilloSignature_AUCELL.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# optional: nicer default sizing/fonts without seaborn
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titleweight"] = "bold"

celltypes = list(signatureDF["celltype"].dropna().unique())
n = len(celltypes)

# panel layout
ncols = min(3, n)
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.8 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, celltype in zip(axes, celltypes):
    xcol = f"scoregenes_{celltype}"
    ycol = f"AUC_score_{celltype}"
    
    if xcol not in adata.obs.columns or ycol not in adata.obs.columns:
        ax.set_visible(False)
        continue

    df = adata.obs[[xcol, ycol]].copy()
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    if df.empty:
        ax.set_visible(False)
        continue

    x = df[xcol].to_numpy()
    y = df[ycol].to_numpy()

    # scatter
    ax.scatter(
        x, y,
        s=16,
        alpha=0.45,
        edgecolors="none"
    )

    # regression line
    if len(df) >= 2 and np.std(x) > 0 and np.std(y) > 0:
        m, b = np.polyfit(x, y, 1)
        xx = np.linspace(x.min(), x.max(), 200)
        yy = m * xx + b
        ax.plot(xx, yy, linewidth=2.2)

        r = np.corrcoef(x, y)[0, 1]
        ax.text(
            0.03, 0.97,
            f"r = {r:.2f}\nn = {len(df)}",
            transform=ax.transAxes,
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="0.8")
        )

    ax.set_title(celltype)
    ax.set_xlabel("ScoreGenes")
    ax.set_ylabel("AUCell")
    ax.grid(True, alpha=0.2)

# hide unused panels
for ax in axes[len(celltypes):]:
    ax.set_visible(False)

fig.suptitle("ScoreGenes vs AUCell by cell type", y=1.02, fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()